# IMC-2025 — XFeat + COLMAP inference

Interactive equivalent of `test.sh`: extracts XFeat features for every test image, matches every pair, runs COLMAP incremental Structure-from-Motion and writes `submission.csv` in the official IMC format.

All code lives in [`imc_xfeat/`](../imc_xfeat) and the data comes from [`data/image-matching/`](../data/image-matching). The notebook just orchestrates the same calls `python -m imc_xfeat.run_submission` makes.

## 1. Setup paths

Change cwd to the project root (one level up from `notebooks/`) so relative paths like `data/image-matching/...` resolve, and add the root to `sys.path` so `import imc_xfeat...` works regardless of where the notebook server was launched.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("project root:", PROJECT_ROOT)

## 2. Configuration

Same knobs as `test.sh`. Edit and re-run this cell to switch GPU, change thresholds, or point at a different weights file.

In [ ]:
GPU_IDS = "0"            

TOP_K          = 4096        # XFeat keypoints per image
MIN_COSSIM     = 0.82        # XFeat match cosine-similarity threshold
MIN_MATCHES    = 15          # keep pair only if it has >= this many matches
MIN_MODEL_SIZE = 3           # minimum images for a COLMAP sub-reconstruction

TEST_DIR          = "data/image-matching/test"
SAMPLE_SUBMISSION = "data/image-matching/sample_submission.csv"
WORK_DIR          = "imc_xfeat/work"
OUTPUT            = "submission.csv"

WEIGHTS = "imc_xfeat/checkpoints/xfeat_imc_latest.pt"
if not Path(WEIGHTS).exists():
    WEIGHTS = "accelerated_features/weights/xfeat.pt"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS
print("using weights:", WEIGHTS)

## 3. Imports

Pull the pipeline functions from the project's `imc_xfeat` package. `pycolmap` is imported up front so missing-deps errors surface immediately.

In [ ]:
import time
import numpy as np
import pycolmap 

from imc_xfeat.imc_io import (
    list_dataset_images, 
    mat_to_str,                          
    read_sample_submission, 
    write_submission
)
from imc_xfeat.reconstruct import (
    extract_features,
    match_pairs,
    reconstruct
)
from imc_xfeat.xfeat_utils import load_xfeat

IDENTITY_R = mat_to_str(np.eye(3))
ZERO_T     = mat_to_str(np.zeros(3))

## 4. Discover test images

List every `<dataset>/<image>` under `TEST_DIR`, and (optionally) load the `image_id` strings from the sample submission so each output row matches the official format.

In [ ]:
datasets = list_dataset_images(TEST_DIR)
assert datasets, f"No <dataset>/<image> folders found under {TEST_DIR}"
print(f"{sum(len(v) for v in datasets.values())} images across {len(datasets)} dataset(s):",
      list(datasets))

image_id_map = {}
if Path(SAMPLE_SUBMISSION).exists():
    image_id_map = {(r['dataset'], r['image']): r['image_id']
                    for r in read_sample_submission(SAMPLE_SUBMISSION)}
    print(f"loaded {len(image_id_map)} image_id mappings from sample submission")

## 5. Load XFeat

In [ ]:
xfeat = load_xfeat(WEIGHTS, top_k=TOP_K)
print("device:", xfeat.dev)

## 6. Run the per-dataset pipeline

For each dataset: XFeat extract → exhaustive match → COLMAP `incremental_mapping`. Images that COLMAP could not register fall back to identity pose + `scene=outliers`.

In [ ]:
rows, t0 = [], time.time()

for dataset, names in datasets.items():
    print(f"\n=== dataset: {dataset}  ({len(names)} images) ===")
    image_dir = os.path.join(TEST_DIR, dataset)

    poses = {}
    try:
        feats = extract_features(xfeat, image_dir, names, TOP_K)
        pairs = match_pairs(xfeat, feats, names, MIN_COSSIM, MIN_MATCHES)
        print(f"  kept {len(pairs)} candidate image pairs")
        poses = reconstruct(image_dir, names, feats, pairs,
                            os.path.join(WORK_DIR, dataset),
                            min_model_size=MIN_MODEL_SIZE)
    except Exception as exc:
        print(f"  ERROR processing {dataset}: {exc}")

    for name in names:
        image_id = image_id_map.get((dataset, name), f"{dataset}_{name}_public")
        if name in poses:
            cluster_idx, R, t = poses[name]
            rows.append({"image_id": image_id, "dataset": dataset,
                         "scene": f"cluster{cluster_idx}", "image": name,
                         "rotation_matrix": mat_to_str(R),
                         "translation_vector": mat_to_str(t)})
        else:
            rows.append({"image_id": image_id, "dataset": dataset,
                         "scene": "outliers", "image": name,
                         "rotation_matrix": IDENTITY_R,
                         "translation_vector": ZERO_T})

print(f"\ntotal: {time.time() - t0:.1f}s, {len(rows)} rows ready")

## 7. Write `submission.csv`

In [ ]:
write_submission(OUTPUT, rows)
posed = sum(1 for r in rows if r['scene'] != 'outliers')
print(f"wrote {OUTPUT}  |  {posed}/{len(rows)} images posed")

## 8. Quick sanity check

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print("scene distribution:")
print(df['scene'].value_counts())
df.head(10)